<a href="https://colab.research.google.com/github/VatsalyaBhadaurya/ACT-Policy-IssacSim/blob/main/ACT_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install LeRobot (includes ACT, PyTorch, etc.)
!pip install -U lerobot

# Optional: Weights & Biases for logging
!pip install wandb
import wandb
wandb.login()  # Enter your API key


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: Paste your API key and hit enter:

 ··········


wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vatbhadaurya (vatbhadaurya-gautam-buddha-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Pre-install HDF5 and build tools (fixes h5py compilation)
!apt update -qq && apt install -y -qq libhdf5-dev hdf5-tools libhdf5-serial-dev h5py

# Upgrade pip and torch first
!pip install --upgrade pip torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

# Clone and install minimal LeRobot (avoids heavy extras like gui)
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e .[train] --no-build-isolation --no-cache-dir

# Verify (ignore warnings)
!python -c "from lerobot.common.datasets.utils import get_dataset_configs; from lerobot.common.policies.act.dataset import ACTDataset; print('LeRobot installed successfully!')"


109 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package h5py
fatal: destination path 'lerobot' already exists and is not an empty directory.
/content/lerobot
Obtaining file:///content/lerobot
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.5.1-0.editable-py3-none-any.whl size=12869 sha256=7cd262f041d1ec50e0264e25a9645409111e95a49d920cd76db6d0b52a4d2a80
  Stored in directory: /tmp/pip-ephem-wheel-cache-8ejihz_6/wheels/09/b4/fe/75732b1d640db96ba1f856f2b7328b232a03b696a39cb59686
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.5.0
    Un

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'lerobot.common'


In [3]:
import lerobot
print("LeRobot version:", lerobot.__version__)
print("LeRobot dir:", dir(lerobot))

# Check if datasets submodule exists
try:
    import lerobot.datasets
    print("datasets submodule:", dir(lerobot.datasets))
except:
    print("No datasets submodule")


LeRobot version: 0.5.1
LeRobot dir: ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'available_cameras', 'available_datasets', 'available_datasets_per_env', 'available_envs', 'available_motors', 'available_policies', 'available_policies_per_env', 'available_real_world_datasets', 'available_robots', 'available_tasks_per_env', 'configs', 'datasets', 'env_dataset_pairs', 'env_dataset_policy_triplets', 'env_task_pairs', 'envs', 'itertools', 'motors', 'optim', 'policies', 'processor', 'robots', 'teleoperators', 'utils']
datasets submodule: ['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'backward_compatibility', 'compute_stats', 'factory', 'image_writer', 'lerobot_dataset', 'streaming_dataset', 'transforms', 'utils', 'video_utils']


In [4]:
from datasets import load_dataset
import torch
import numpy as np

# Load LeRobot HF dataset directly
repo_id = "lerobot/aloha_sim_insertion_human"
hf_dataset = load_dataset(repo_id, split="train")

print(f"Dataset loaded: {len(hf_dataset)} episodes")
print("Sample keys:", hf_dataset[0].keys())
print("FPS:", hf_dataset.info.description)


README.md: 0.00B [00:00, ?B/s]

data/chunk-000/file-000.parquet:   0%|          | 0.00/856k [00:00<?, ?B/s]

data/chunk-000/file-001.parquet:   0%|          | 0.00/848k [00:00<?, ?B/s]

data/chunk-000/file-002.parquet:   0%|          | 0.00/856k [00:00<?, ?B/s]

data/chunk-000/file-003.parquet:   0%|          | 0.00/293k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset loaded: 25000 episodes
Sample keys: dict_keys(['observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.done', 'index', 'task_index'])
FPS: 


In [8]:
def create_act_dataloader(hf_dataset, batch_size=16, horizon_len=16):
    def collate_fn(batch):
        # Extract core observation and action data
        states = torch.stack([torch.tensor(ep['observation.state']) for ep in batch]).float()
        actions = torch.stack([torch.tensor(ep['action']) for ep in batch]).float()

        return {
            "observation": {"state": states},
            "action": actions
        }

    return torch.utils.data.DataLoader(
        hf_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

# Test it ✅
dataloader = create_act_dataloader(hf_dataset, batch_size=8)
sample_batch = next(iter(dataloader))
print("✅ Dataloader perfect:")
print(f"  observation.state: {sample_batch['observation']['state'].shape}")  # [8, 14]
print(f"  action: {sample_batch['action'].shape}")                          # [8, 14]


✅ Dataloader perfect:
  observation.state: torch.Size([8, 14])
  action: torch.Size([8, 14])


In [9]:
import torch.optim as optim
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

class ACTPolicy(nn.Module):
    def __init__(self, state_dim=14, action_dim=14, horizon_len=16, hidden_dim=256):
        super().__init__()
        self.state_encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Transformer for action chunking
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.action_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim * horizon_len)
        )
        self.horizon_len = horizon_len

    def forward(self, obs):
        # obs: {"state": [B, 14]}
        x = self.state_encoder(obs["state"])

        # Predict action chunk
        x = self.transformer(x.unsqueeze(1)).squeeze(1)
        actions = self.action_head(x)  # [B, 14*16]
        return actions.view(-1, self.horizon_len, 14)  # [B, 16, 14]

# Initialize
model = ACTPolicy()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.MSELoss()

print(f"✅ Model ready: {sum(p.numel() for p in model.parameters())} params")


✅ Model ready: 5453792 params


In [11]:
# ✅ CORRECTED TRAINING LOOP
model = ACTPolicy()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.MSELoss()

print("Starting training...")
model.train()

# Iterate EPOCHS, then BATCHES per epoch
for epoch in range(10):  # 10 epochs
    total_loss = 0
    num_batches = 0

    # Single pass through entire dataset per epoch
    for batch_idx, batch in enumerate(dataloader):
        optimizer.zero_grad()

        # Forward pass
        pred_actions = model(batch["observation"])
        target_actions = batch["action"]  # [B, 14] - single timestep

        # ACT predicts horizon, but target is single step (standard behavior)
        loss = criterion(pred_actions[:, 0, :], target_actions)  # Use first timestep

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        # Print every 50 batches
        if batch_idx % 50 == 0:
            avg_loss = total_loss / num_batches
            print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {avg_loss:.4f}")

    # Epoch summary
    epoch_loss = total_loss / num_batches
    print(f"✅ Epoch {epoch} COMPLETE - Avg Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), "./act_policy_aloha.pt")
print("🎉 Training finished! Model saved.")


Starting training...
Epoch 0, Batch 0, Loss: 0.1412
Epoch 0, Batch 50, Loss: 0.0314
Epoch 0, Batch 100, Loss: 0.0218
Epoch 0, Batch 150, Loss: 0.0177
Epoch 0, Batch 200, Loss: 0.0150
Epoch 0, Batch 250, Loss: 0.0133
Epoch 0, Batch 300, Loss: 0.0120
Epoch 0, Batch 350, Loss: 0.0110
Epoch 0, Batch 400, Loss: 0.0102
Epoch 0, Batch 450, Loss: 0.0097
Epoch 0, Batch 500, Loss: 0.0092
Epoch 0, Batch 550, Loss: 0.0087
Epoch 0, Batch 600, Loss: 0.0083
Epoch 0, Batch 650, Loss: 0.0080
Epoch 0, Batch 700, Loss: 0.0077
Epoch 0, Batch 750, Loss: 0.0074
Epoch 0, Batch 800, Loss: 0.0072
Epoch 0, Batch 850, Loss: 0.0070
Epoch 0, Batch 900, Loss: 0.0068
Epoch 0, Batch 950, Loss: 0.0066
Epoch 0, Batch 1000, Loss: 0.0064
Epoch 0, Batch 1050, Loss: 0.0063
Epoch 0, Batch 1100, Loss: 0.0062
Epoch 0, Batch 1150, Loss: 0.0060
Epoch 0, Batch 1200, Loss: 0.0059
Epoch 0, Batch 1250, Loss: 0.0057
Epoch 0, Batch 1300, Loss: 0.0056
Epoch 0, Batch 1350, Loss: 0.0055
Epoch 0, Batch 1400, Loss: 0.0054
Epoch 0, Batch 1

In [12]:
# Verify model works
model.eval()
with torch.no_grad():
    test_pred = model(sample_batch["observation"])
    print(f"✅ Prediction shape: {test_pred.shape}")  # [8, 16, 14]


✅ Prediction shape: torch.Size([8, 16, 14])
